# Répondre aux questions sur la presse de 1914

Un assistant qui répond aux questions à partir des 64 000 passages de la presse de 1914, en citant ses sources (un RAG), et son évaluation sur 20 questions dont on connaît la réponse et le ou les passages qui la donnent.

1. La recherche : hybride, puis reclassement (les briques du TP de recherche de produits)
2. Évaluer la recherche
3. Générer une réponse sourcée, ou refuser
4. Régler la recherche par la mesure
5. Bonus : un LLM juge la fidélité des réponses

Ce notebook va plus vite avec un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Sans GPU, il tourne quand même, en plus petit : la recherche n'est évaluée que sur 4 questions au lieu de 20, et un LLM plus petit (Qwen3-1.7B au lieu de Qwen3-4B) ne répond qu'à 2 questions (l'une a une réponse, l'autre non).

## Données et modèles

Les passages, leurs *embeddings* déjà calculés (512 dimensions), les questions ; Ollama s'installe et télécharge le modèle en arrière-plan.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-qa/questions.jsonl
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q bm25s

import os
import subprocess

import torch

GPU = torch.cuda.is_available()
# Sur GPU, Qwen3-4B. Sur CPU, Qwen3-1.7B, deux fois plus rapide : un modèle « hybride »,
# qui réfléchit avant de répondre, sauf avec reasoning_effort="none" (dans les appels).
MODELE = "qwen3:4b-instruct-2507-q4_K_M" if GPU else "qwen3:1.7b"
# Sur GPU, 8 demandes à la fois ; sur CPU, une : le début commun des prompts (les
# consignes) n'est alors calculé qu'une fois.
paralleles = "8" if GPU else "1"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": paralleles, "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(
  ["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
telechargement = subprocess.Popen(
  f"sleep 5 && ollama pull {MODELE}",
  shell=True,
  stdout=subprocess.DEVNULL,
  stderr=subprocess.DEVNULL,
)

## 1. La recherche

Les fonctions de recherche sont données : ce sont celles du TP de recherche de produits. `rechercher(question, k)` renvoie les indices des `k` meilleurs passages : BM25 et recherche dense (100 candidats chacun), fusionnés par RRF, puis les `A_RECLASSER` premiers (30) reclassés par le cross-encodeur (`reclasser=False` pour s'en passer). Chaque question n'est cherchée qu'une fois : le classement est gardé en mémoire, quel que soit `k`. Lancez la cellule (une à deux minutes, durée indicative : elle attend aussi le LLM, et le charge sur le GPU avant les autres modèles), et essayez une question.

In [ ]:
# bm25s se sert de JAX s'il est installé (c'est le cas sur Colab), et JAX réserve d'office
# 75 % de la mémoire du GPU : on le cantonne au CPU, le GPU est pour les modèles.
os.environ["JAX_PLATFORMS"] = "cpu"

from collections import defaultdict
from functools import cache
from concurrent.futures import ThreadPoolExecutor

import bm25s
import torch
import numpy as np
import pandas as pd
from openai import LengthFinishReasonError, OpenAI
from pydantic import BaseModel
from sentence_transformers import CrossEncoder, SentenceTransformer

telechargement.wait()  # le modèle est téléchargé
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# Le LLM d'abord : un premier appel le charge (sur le GPU s'il y en a un), où il reste, avant les autres modèles.
client.chat.completions.create(
  model=MODELE, messages=[{"role": "user", "content": "Bonjour"}], max_tokens=1
)
print(MODELE, "prêt")

passages = pd.read_parquet("passages.parquet")
vecteurs = np.load("embeddings-512.npy").astype(np.float32)
questions = pd.read_json("questions.jsonl", lines=True)
# Une question sans réponse a `answer` à null : pandas en fait un NaN, que `or` prend
# pour une valeur. On le remplace une fois pour toutes par None.
questions["answer"] = questions.answer.astype(object).where(
  questions.answer.notna(), None
)

DEVICE = "cuda" if GPU else "cpu"
# Sur CPU, float32 : les demi-précisions y sont très lentes.
DTYPE = torch.float16 if GPU else torch.float32
encodeur = SentenceTransformer(
  "Qwen/Qwen3-Embedding-0.6B",
  device=DEVICE,
  truncate_dim=512,
  model_kwargs={"dtype": DTYPE},
)
reclasseur = CrossEncoder(
  "BAAI/bge-reranker-v2-m3",
  device=DEVICE,
  max_length=512,
  model_kwargs={"dtype": DTYPE},
)
# Les candidats que le cross-encodeur reclasse. Sans GPU, chacun prend deux à trois
# secondes, mais avec moins de candidats, il n'aurait plus rien à remonter : on y évalue
# moins de questions (exercice 2), pas moins de candidats.
A_RECLASSER = 30
index_mots = bm25s.BM25()
index_mots.index(
  bm25s.tokenize(passages.text.tolist(), show_progress=False), show_progress=False
)


def rrf(*classements, k: int = 60) -> list[int]:
  scores = defaultdict(float)
  for classement in classements:
    for rang, doc in enumerate(classement, start=1):
      scores[doc] += 1 / (k + rang)
  return sorted(scores, key=scores.get, reverse=True)


@cache
def classer(question: str, reclasser: bool = True) -> list[int]:
  lexicale = index_mots.retrieve(
    bm25s.tokenize([question], show_progress=False), k=100, show_progress=False
  )[0][0]
  requete = encodeur.encode_query(question).astype(np.float32)
  requete /= np.linalg.norm(requete)
  dense = np.argsort(-(vecteurs @ requete))[:100]
  candidats = rrf(lexicale.tolist(), dense.tolist())
  if not reclasser:
    return candidats
  premiers = candidats[:A_RECLASSER]
  scores = reclasseur.predict(
    [(question, passages.text[i]) for i in premiers], batch_size=32
  )
  return [premiers[i] for i in np.argsort(-scores)]


def rechercher(question: str, k: int = 5, reclasser: bool = True) -> list[int]:
  # Mis en cache : chaque question n'est cherchée qu'une fois, quel que soit k.
  return classer(question, reclasser)[:k]


for i in rechercher("Où Jaurès a-t-il été assassiné ?", k=3):
  print(passages.id[i], passages.text[i][:200])

## 2. Évaluer la recherche

Pour chacune des questions de `evaluees` (les 20 ; sans GPU, 4) qui ont une réponse dans le corpus (18 des 20), `passage_ids` donne les passages qui la contiennent (un seul, le plus souvent). Écrivez `rappel_a(k, reclasser)` : la part des questions dont un passage attendu est parmi les `k` premiers résultats. Comparez avec et sans reclassement, à `k=5`.

**Documentation utile**

- [`Series.str.len`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.len.html)
- [`DataFrame.itertuples`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.itertuples.html)
- [`set`](https://docs.python.org/3/library/stdtypes.html#set-types-set-frozenset)

In [ ]:
# Les questions évaluées : les 20 sur GPU ; sans GPU, 4 (cinq minutes environ, durée
# indicative : le cross-encodeur y lit chaque passage en deux à trois secondes), dont
# deux que le reclassement remonte de loin.
evaluees = questions if GPU else questions.iloc[[1, 2, 3, 5]]

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `evaluees.passage_ids.str.len() > 0` : les questions qui ont une réponse (`str.len()` marche aussi sur une colonne de listes)
- `rechercher(question, k, reclasser)` renvoie les positions des passages trouvés ; `passages.id[positions]` donne leurs identifiants
- `set(trouvés) & set(ligne.passage_ids)` : l'intersection des passages trouvés et attendus ; `bool(...)` vaut `True` si elle n'est pas vide
- `avec_reponse.itertuples()` : une ligne par question, avec `ligne.question` et `ligne.passage_ids` ; `np.mean(succes)` : la part de succès

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
avec_reponse = evaluees[evaluees.passage_ids.str.len() > ...]


def rappel_a(k: int = 5, reclasser: bool = True) -> float:
  succes = [
    bool(set(passages.id[rechercher(ligne.question, k, reclasser)]) & set(...))
    for ligne in avec_reponse.itertuples()
  ]
  return float(np.mean(succes))


for reclasser in (False, True):
  print(
    f"reclassement={reclasser} : {rappel_a(5, reclasser):.0%} des passages attendus dans les 5 premiers"
  )
```

### Solution

In [ ]:
avec_reponse = evaluees[evaluees.passage_ids.str.len() > 0]


def rappel_a(k: int = 5, reclasser: bool = True) -> float:
  succes = [
    bool(
      set(passages.id[rechercher(ligne.question, k, reclasser)])
      & set(ligne.passage_ids)
    )
    for ligne in avec_reponse.itertuples()
  ]
  return float(np.mean(succes))


for reclasser in (False, True):
  print(
    f"reclassement={reclasser} : {rappel_a(5, reclasser):.0%} des passages attendus dans les 5 premiers"
  )

## 3. Générer une réponse sourcée, ou refuser

Écrivez `repondre(question)` qui :

- retrouve les 5 meilleurs passages, numérotés de 1 à 5 dans le prompt, avec leur date et leur journal
- demande au modèle une réponse au format `Reponse` (ci-dessous) : la réponse, les numéros des passages qui la justifient, et `trouve=False` si les passages ne contiennent pas la réponse
- consigne : ne répondre **qu'à partir des passages**, en français, en une ou deux phrases

Lancez-la sur les questions de `posees` en parallèle (les 20 ; sans GPU, 2 : l'une a une réponse, l'autre non), et comparez ses réponses aux réponses de référence. Les deux questions sans réponse sont-elles refusées ?

**Documentation utile**

- [`BaseModel.model_dump`](https://docs.pydantic.dev/latest/api/base_model/#pydantic.BaseModel.model_dump)
- [`enumerate`](https://docs.python.org/3/library/functions.html#enumerate)
- [Formats de date](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes)
- [`ThreadPoolExecutor.map`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map)

In [ ]:
class Reponse(BaseModel):
  trouve: bool
  reponse: str
  citations: list[int]


# Les questions posées au LLM : les 20 sur GPU ; sans GPU, 2 (cinq minutes environ,
# durée indicative), dont une sans réponse.
posees = questions if GPU else questions.iloc[[3, 18]]

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `rechercher(question, k)` renvoie les positions des `k` meilleurs passages ; `enumerate(extraits, start=1)` les numérote à partir de 1
- `f"[{n}] ({passages.newspaper[i]}, {passages.date[i]:%d/%m/%Y}) {passages.text[i]}"` : un passage avec son numéro, son journal et sa date (`%d/%m/%Y` : jour/mois/année) ; les passages sont joints par des lignes vides, puis la question
- `client.chat.completions.parse(model=MODELE, messages=[{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": ...}], response_format=Reponse, temperature=0, reasoning_effort="none", max_tokens=300)`
- `reasoning_effort="none"` : le modèle répond sans phase de réflexion (sans GPU, Qwen3-1.7B en aurait une, qui multiplie le temps de réponse)
- `max_tokens=300` : borne la réponse. Un petit modèle boucle parfois dans son JSON jusqu'à la limite, et `parse` lève alors `LengthFinishReasonError` (à importer d'`openai`) : comptez ce cas comme un refus, avec `trouve` à `False`
- `retour.choices[0].message.parsed.model_dump()` : l'objet lu, converti en dictionnaire ; ajoutez-y `"sources"`, les identifiants `passages.id[extraits[n - 1]]` des passages cités (`n` dans `citations`, entre 1 et `len(extraits)`)
- `ThreadPoolExecutor(8)` : 8 appels en parallèle ; `list(executeur.map(repondre, posees.question))` renvoie les réponses dans l'ordre
- les consignes (`INSTRUCTIONS`) : répondre **uniquement** à partir des passages numérotés, en français, en une ou deux phrases ; mettre les numéros des passages dans `citations` ; si les passages ne contiennent pas la réponse, `trouve` à `false` sans rien inventer

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
INSTRUCTIONS = """Tu réponds aux questions sur la presse française de 1914, uniquement à partir
des passages fournis, numérotés. Réponds en français, en une ou deux phrases, et mets dans
citations les numéros des passages qui justifient ta réponse (pas dans la réponse elle-même).
Si les passages ne contiennent pas la réponse, mets trouve à false et n'invente rien."""


def repondre(question: str, k: int = 5) -> dict:
  extraits = rechercher(question, k)
  contexte = "\n\n".join(
    f"[{n}] ({passages.newspaper[i]}, {passages.date[i]:%d/%m/%Y}) {passages.text[i]}"
    for n, i in enumerate(extraits, start=...)
  )
  try:
    retour = client.chat.completions.parse(
      model=MODELE,
      messages=[
        {"role": "system", "content": ...},
        {"role": "user", "content": f"Passages :\n{contexte}\n\nQuestion : {question}"},
      ],
      response_format=...,
      temperature=0,
      reasoning_effort="none",
      max_tokens=300,
    )
  except LengthFinishReasonError:
    # Un petit modèle boucle parfois dans son JSON : la réponse compte pour un refus.
    return {"trouve": ..., "reponse": "", "citations": [], "sources": []}
  resultat = retour.choices[0].message.parsed.model_dump()
  resultat["sources"] = [
    passages.id[extraits[n - 1]]
    for n in resultat["citations"]
    if 1 <= n <= len(extraits)
  ]
  return resultat


with ThreadPoolExecutor(8) as executeur:
  reponses = list(executeur.map(..., posees.question))
for ligne, resultat in zip(posees.itertuples(), reponses):
  print(
    f"Q{ligne.id} {ligne.question}\n  référence : {ligne.answer or '(aucune)'}\n  réponse   : {resultat['reponse'] if resultat['trouve'] else '(refus)'}\n"
  )
```

### Solution

In [ ]:
INSTRUCTIONS = """Tu réponds aux questions sur la presse française de 1914, uniquement à partir
des passages fournis, numérotés. Réponds en français, en une ou deux phrases, et mets dans
citations les numéros des passages qui justifient ta réponse (pas dans la réponse elle-même).
Si les passages ne contiennent pas la réponse, mets trouve à false et n'invente rien."""


def repondre(question: str, k: int = 5) -> dict:
  extraits = rechercher(question, k)
  contexte = "\n\n".join(
    f"[{n}] ({passages.newspaper[i]}, {passages.date[i]:%d/%m/%Y}) {passages.text[i]}"
    for n, i in enumerate(extraits, start=1)
  )
  try:
    retour = client.chat.completions.parse(
      model=MODELE,
      messages=[
        {"role": "system", "content": INSTRUCTIONS},
        {"role": "user", "content": f"Passages :\n{contexte}\n\nQuestion : {question}"},
      ],
      response_format=Reponse,
      temperature=0,
      reasoning_effort="none",
      max_tokens=300,
    )
  except LengthFinishReasonError:
    # Un petit modèle boucle parfois dans son JSON : la réponse compte pour un refus.
    return {"trouve": False, "reponse": "", "citations": [], "sources": []}
  resultat = retour.choices[0].message.parsed.model_dump()
  resultat["sources"] = [
    passages.id[extraits[n - 1]]
    for n in resultat["citations"]
    if 1 <= n <= len(extraits)
  ]
  return resultat


with ThreadPoolExecutor(8) as executeur:
  reponses = list(executeur.map(repondre, posees.question))
for ligne, resultat in zip(posees.itertuples(), reponses):
  print(
    f"Q{ligne.id} {ligne.question}\n  référence : {ligne.answer or '(aucune)'}\n  réponse   : {resultat['reponse'] if resultat['trouve'] else '(refus)'}\n"
  )

## 4. Régler la recherche par la mesure

Un RAG se règle comme un modèle : en mesurant. Le taux de l'exercice 2 dit combien de questions passent, pas où se trouve leur passage. Écrivez `rang_attendu(ligne, reclasser)` : pour une question (une ligne de `avec_reponse`), le rang du premier passage attendu, à partir de 1, ou `None` s'il n'est pas dans le classement. Affichez-le pour chaque question, avec et sans reclassement : que change le reclassement ? Déduisez-en le plus petit `k` qui couvre les questions retrouvées. Quel `k` choisiriez-vous, sachant que chaque passage ajouté allonge le prompt (coût, latence) ? Pourquoi un `k` plus grand ne rattraperait-il pas une question à `None` ?

**Documentation utile**

- [`enumerate`](https://docs.python.org/3/library/functions.html#enumerate)
- [`max`](https://docs.python.org/3/library/functions.html#max)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `classer(question, reclasser)` : le classement entier, dont `rechercher` ne garde que les `k` premiers ; avec reclassement, il se limite aux `A_RECLASSER` candidats reclassés
- `passages.id[positions]` : les identifiants des passages, dans l'ordre du classement
- `enumerate(trouves, start=1)` : chaque identifiant avec son rang ; `ligne.passage_ids` : les passages attendus
- `rang_attendu(ligne)` : le rang avec reclassement (`reclasser=True`, la valeur par défaut) ; le `k` cherché est le plus grand de ces rangs, sans les `None`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def rang_attendu(ligne, reclasser: bool = True) -> int | None:
  trouves = passages.id[classer(ligne.question, ...)]
  for rang, id_passage in enumerate(trouves, start=...):
    if id_passage in ...:
      return rang
  return None


for ligne in avec_reponse.itertuples():
  print(
    f"Q{ligne.id} : sans reclassement {rang_attendu(ligne, ...)}, "
    f"avec {rang_attendu(ligne)}"
  )

rangs = [rang_attendu(ligne) for ligne in avec_reponse.itertuples()]
retrouvees = [rang for rang in rangs if rang is not ...]
print(
  f"k={max(...)} couvre les {len(retrouvees)} questions retrouvées sur {len(avec_reponse)}"
)
```

### Solution

In [ ]:
def rang_attendu(ligne, reclasser: bool = True) -> int | None:
  trouves = passages.id[classer(ligne.question, reclasser)]
  for rang, id_passage in enumerate(trouves, start=1):
    if id_passage in ligne.passage_ids:
      return rang
  return None


for ligne in avec_reponse.itertuples():
  print(
    f"Q{ligne.id} : sans reclassement {rang_attendu(ligne, False)}, "
    f"avec {rang_attendu(ligne)}"
  )

rangs = [rang_attendu(ligne) for ligne in avec_reponse.itertuples()]
retrouvees = [rang for rang in rangs if rang is not None]
print(
  f"k={max(retrouvees)} couvre les {len(retrouvees)} questions retrouvées "
  f"sur {len(avec_reponse)}"
)

# Le reclassement remonte des passages que la fusion classait loin : c'est lui qui permet
# un k plus petit, donc un prompt plus court. Une question à None n'est pas parmi les
# A_RECLASSER candidats : aucun k ne la rattrape. Elle se corrige en amont, dans la recherche,
# pas en donnant plus de passages au LLM.

## 5. Bonus : un LLM juge la fidélité

Pour chaque réponse (celles des questions de `posees`), demandez au modèle de juger, à partir des passages cités et de la réponse de référence : la réponse est-elle **fidèle** aux passages (rien d'inventé) ? Est-elle **juste** ? Comptez. Pourquoi faut-il se méfier d'un modèle qui juge ses propres réponses ?

**Documentation utile**

- [`ThreadPoolExecutor.map`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map)
- [`DataFrame.set_index`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `reasoning_effort="none"` : le modèle répond sans phase de réflexion (sans GPU, Qwen3-1.7B en aurait une, qui multiplie le temps de réponse)
- `class Verdict(BaseModel)` : deux champs booléens, `fidele` et `juste`
- `passages.set_index("id").text[source]` : le texte d'un passage d'après son identifiant ; les passages cités sont les `resultat["sources"]`
- le prompt : la question, la réponse de référence (`ligne.answer`, `None` si la question est sans réponse), les passages cités, la réponse à juger, la définition de `fidele` (rien que les passages ne disent) et de `juste` (même sens que la référence ; un refus est juste si la question est sans réponse)
- rappelez le format dans le prompt (`Réponds par un objet JSON : {"fidele": true ou false, "juste": true ou false}`) : le modèle ne voit pas le schéma, et juge moins bien sans
- `executeur.map(juge, posees.question, posees.answer, reponses)` : avec plusieurs itérables, `juge(question[i], answer[i], reponses[i])` est appelée pour chaque `i`
- `pd.DataFrame(verdicts).mean()` : la part de `True` dans chaque colonne

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class Verdict(BaseModel):
  fidele: bool
  juste: ...


def juge(question: str, reference: str | None, resultat: dict) -> dict:
  cites = "\n\n".join(passages.set_index("id").text[source] for source in ...)
  retour = client.chat.completions.parse(
    model=MODELE,
    messages=[
      {
        "role": "user",
        "content": (
          f"Question : {question}\nRéponse de référence : {reference or 'aucune, la question est sans réponse'}\n"
          f"Passages cités :\n{cites or '(aucun)'}\n\nRéponse à juger : "
          f"{resultat['reponse'] if resultat['trouve'] else '(refus)'}\n\n"
          "fidele : la réponse ne dit rien que les passages cités ne disent. "
          "juste : la réponse dit la même chose que la référence (un refus est juste si la question est sans réponse).\n"
          'Réponds par un objet JSON : {"fidele": true ou false, "juste": true ou false}'
        ),
      }
    ],
    response_format=...,
    temperature=0,
    reasoning_effort="none",
  )
  return retour.choices[0].message.parsed.model_dump()


with ThreadPoolExecutor(8) as executeur:
  verdicts = list(executeur.map(juge, ..., ..., reponses))
print(pd.DataFrame(verdicts).mean())
```

### Solution

In [ ]:
class Verdict(BaseModel):
  fidele: bool
  juste: bool


def juge(question: str, reference: str | None, resultat: dict) -> dict:
  cites = "\n\n".join(
    passages.set_index("id").text[source] for source in resultat["sources"]
  )
  retour = client.chat.completions.parse(
    model=MODELE,
    messages=[
      {
        "role": "user",
        "content": (
          f"Question : {question}\nRéponse de référence : {reference or 'aucune, la question est sans réponse'}\n"
          f"Passages cités :\n{cites or '(aucun)'}\n\nRéponse à juger : "
          f"{resultat['reponse'] if resultat['trouve'] else '(refus)'}\n\n"
          "fidele : la réponse ne dit rien que les passages cités ne disent. "
          "juste : la réponse dit la même chose que la référence (un refus est juste si la question est sans réponse).\n"
          # Le format rappelé : le modèle ne voit pas le schéma, et juge moins bien sans.
          'Réponds par un objet JSON : {"fidele": true ou false, "juste": true ou false}'
        ),
      }
    ],
    response_format=Verdict,
    temperature=0,
    reasoning_effort="none",
  )
  return retour.choices[0].message.parsed.model_dump()


with ThreadPoolExecutor(8) as executeur:
  verdicts = list(executeur.map(juge, posees.question, posees.answer, reponses))
print(pd.DataFrame(verdicts).mean())

# Le même modèle génère et juge : il tend à approuver ses propres réponses (biais
# d'auto-préférence). En pratique, on juge avec un autre modèle, plus fort, et on vérifie
# un échantillon des verdicts à la main.